In [157]:
# Mahjong CONSTS
from mahjong.constants import EAST, SOUTH, WEST, NORTH, HAKU, HATSU, CHUN #former 4 are .WINDS, and all 7 are .HONOR_INDICES
from mahjong.constants import FIVE_RED_MAN, FIVE_RED_PIN, FIVE_RED_SOU # Red Dora number cards > .AKA_DORA_LIST
from mahjong.constants import DISPLAY_WINDS # Str display of the winds

# Mahjong methods/classes
from mahjong.hand_calculating.hand import HandCalculator
from mahjong.hand_calculating.hand_config import HandConfig, OptionalRules
from mahjong.meld import Meld
from mahjong.tile import TilesConverter

# imports
import random
import re
from typing import List, Optional, Tuple, Union
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets
import copy
import os
from pathlib import Path
import base64

In [158]:
TOTAL_CARD_NUM = 136 # 4 * [9pin + 9sozu + 9manzu + 3sanngennpai + 4jihai]
TILE_IMAGE_DIR = "./tile_imgs/"

In [159]:
# UTILITY FUNCTIONS

class MahjongConverter:
    """
    Utility class for Mahjong tile data conversions.
    All methods are static to serve as a functional engine for wrapper classes.
    """

    @staticmethod
    def to_34_array(tiles_136: List[int]) -> List[int]:
        tiles_34 = [0] * 34
        for t in tiles_136:
            tiles_34[t // 4] += 1
        return tiles_34

    @staticmethod
    def from_34_to_136(tiles_34: List[int]) -> List[int]:
        tiles_136 = []
        for tile_type, count in enumerate(tiles_34):
            base_id = tile_type * 4
            for i in range(min(count, 4)):
                tiles_136.append(base_id + i)
        return tiles_136

    @staticmethod
    def to_str(tiles_136: List[int]) -> str:
        if not tiles_136: return ""
        t34 = MahjongConverter.to_34_array(tiles_136)
        sections = [(0, 9, "m"), (9, 18, "p"), (18, 27, "s"), (27, 31, "z"), (31, 34, "d")]
        res = ""
        for start, end, suffix in sections:
            group = "".join(str(i - start + 1) * t34[i] for i in range(start, end) if t34[i] > 0)
            if group: res += f"{group}{suffix}"
        return res

    @staticmethod
    def to_136(hand_str: str) -> List[int]:
        t34 = [0] * 34
        suites = re.findall(r"(\d+)([mpsz])", hand_str)
        mapping = {"m": 0, "p": 9, "s": 18, "z": 27}
        for digits, suffix in suites:
            for d in digits:
                idx = (int(d) - 1) + mapping[suffix]
                if t34[idx] < 4: t34[idx] += 1
        return MahjongConverter.from_34_to_136(t34)

class MahjongBase:
    """Provides shared comparison and arithmetic logic."""
    def to_34(self) -> List[int]:
        raise NotImplementedError

    def to_ids(self) -> List[int]:
        raise NotImplementedError

    def __eq__(self, other: object) -> bool:
        if isinstance(other, (str, list, int)):
            if isinstance(other, str): other = MSPZD(other)
            elif isinstance(other, list): other = Hand136(other)
            elif isinstance(other, int): other = Hand136([other])
        
        if hasattr(other, 'to_34'):
            return self.to_34() == other.to_34()
        return False

    def __add__(self, other: Union['MahjongBase', str, list, int]) -> 'Hand136':
        """Allows combining hands using the + operator."""
        # Convert 'self' IDs
        self_ids = self.to_ids()
        
        # Convert 'other' to IDs
        if isinstance(other, MahjongBase):
            other_ids = other.to_ids()
        elif isinstance(other, str):
            other_ids = MahjongConverter.to_136(other)
        elif isinstance(other, list):
            other_ids = other
        elif isinstance(other, int):
            other_ids = [other]
        else:
            raise TypeError(f"Unsupported operand type for +: 'MahjongBase' and '{type(other).__name__}'")
            
        return Hand136(self_ids + other_ids)

    def __radd__(self, other: Union[str, list, int]) -> 'Hand136':
        """Handles cases where raw types are on the left side of +."""
        return self.__add__(other)

class Hand136(MahjongBase):
    def __init__(self, ids: Union[int, List[int]]):
        # Flatten list if needed and sort for consistency
        self.ids = sorted(ids) if isinstance(ids, list) else [ids]

    def to_34(self) -> List[int]:
        return MahjongConverter.to_34_array(self.ids)

    def to_ids(self) -> List[int]:
        return self.ids

    def to_mspzd(self) -> 'MSPZD':
        return MSPZD(MahjongConverter.to_str(self.ids))
    
    def remove(self, id):
        self.ids.remove(id)
    
    def __repr__(self):
        return f"Hand136({self.ids})"
    
    def __len__(self) -> int:
        """Allows calling len(hand_obj)"""
        return len(self.ids)

    def __iter__(self):
        """Allows for tile in hand_obj: ..."""
        return iter(self.ids)
    
    def __getitem__(self, position):
        """
        Allows for calling via index
        """
        return self.ids[position]

class MSPZD(MahjongBase):
    def __init__(self, notation: str):
        self.notation = notation

    def to_34(self) -> List[int]:
        return MahjongConverter.to_34_array(MahjongConverter.to_136(self.notation))

    def to_ids(self) -> List[int]:
        return MahjongConverter.to_136(self.notation)

    def to_136(self) -> Hand136:
        return Hand136(self.to_ids())

    def __repr__(self):
        return f"MSPZD('{self.notation}')"

def make_random_hand() -> Hand136:
    """Random generation of tiles of size 13 and 1 drawn tile

    Returns
    -------
    List[Hand136]
        Hand136 len==13, Hand136 len==1
    """
    all_tiles_i = list(range(TOTAL_CARD_NUM))
    random.shuffle(all_tiles_i)
    hand_tiles = sorted(all_tiles_i[:13])
    drawn_tile = all_tiles_i[13]
    return Hand136(hand_tiles), Hand136(drawn_tile)


In [160]:
# GAME SIMULTION CLASS
class MahjongPlayer:
    def __init__(self, name: str, initial_score: int = 35000):
        self.name = name
        self.score = initial_score
        self.hand: Optional[Hand136] = None  # Current hidden tiles
        self.discards: List[Hand136] = []    # History of 136-IDs discarded
        self.melds: List[str] = []           # Calls like 'pon', 'chi', 'kan'
        self.is_riichi = False
        self.is_dealer = False
        self.wind: int #0=East, 1=South, 2=West, 3=North

    def decide_discard(self) -> Hand136:
        discard_tile = Hand136(random.choice(self.hand))
        return discard_tile

    def discard_tile(self, tile: Hand136):
        # Implementation to remove from self.hand and add to self.discards
        self.hand.remove(tile)
        self.discards.append(tile)
        return tile
    
    def reset(self):
        self.hand = None
        self.discards = []
        self.melds = []
        self.is_riichi = False
        self.is_dealer = False
        self.wind = None

    def __repr__(self):
        dealer_mark = " (Dealer)" if self.is_dealer else ""
        return f"[{self.name}]{dealer_mark} Score: {self.score}"
    
class MahjongTable:
    BA = ["East", "South", "West", "North"]

    def __init__(
            self,
            starting_score: int = 25000,
            ending_score: int = 30000,
            num_players: int = 4,
            player_names: list[str] = [],
            total_wind: int = 2 # 1: Tonpuu (4 rounds), 2: Hanchan (8 rounds)
            ):
        self.starting_score = starting_score
        self.ending_score = ending_score
        self.num_players = num_players
        self.total_wind = total_wind
        
        if not player_names:
            self.player_names = [f"P{i}" for i in range(self.num_players)]
        else:
            self.player_names = player_names

        # Initialize players once at the table level
        self.players = [MahjongPlayer(name, self.starting_score) for name in self.player_names]
        
        self.wall: List[Hand136] = []
        self.dead_wall: List[Hand136] = []
        self.dora_indicators: List[Hand136] = []

        # trackers for game state and wind
        self.turn_index: int = 0
        self.num_wind: int = 0
        self.num_kyoku: int = 0
        
        self.renchan = 0
        self.log = []

    @property
    def tiles_remaining(self) -> int:
        return len(self.wall)

    def is_game_over(self) -> bool:
        return self.num_wind >= self.total_wind

    def is_wind_over(self) -> bool:
        return self.num_kyoku >= self.num_players
    
    def is_kyoku_over(self) -> bool:
        # In Riichi, the game stops when the wall hits 14 tiles (the dead wall)
        return self.tiles_remaining <= 0

    def setup_new_kyoku(self):
        self.wall = [Hand136(n) for n in range(136)]
        random.shuffle(self.wall)
        self.dead_wall = [self.wall.pop() for _ in range(14)]
        self.dora_indicators = [self.dead_wall[0]]

        dealer_i = self.num_kyoku
        self.turn_index = dealer_i # Dealer always starts

        for i, p in enumerate(self.players):
            p.reset() 
            p.wind = (i - dealer_i) % self.num_players
            p.is_dealer = (i == dealer_i)
            p.hand = Hand136([self.wall.pop().ids[0] for _ in range(13)])
            
        self._log_state("SETUP ROUND")

    def end_kyoku(self, dealer_renchan: bool):
        """Single point of truth for progressing game state."""
        if dealer_renchan:
            self.renchan += 1
        else:
            self.renchan = 0
            self.num_kyoku += 1
            
            if self.is_wind_over():
                self.num_wind += 1
                self.num_kyoku = 0

    def next_turn(self):
        self.turn_index = (self.turn_index + 1) % self.num_players

    def play_game(self):
        """Main loop: progresses until the total number of winds is reached."""
        while not self.is_game_over():
            self.play_wind()
            # Note: num_wind is now handled inside end_kyoku
            self._log_state("BA OVER")
        self._log_state("GAME OVER")
    
    def play_wind(self):
        """Plays through one full rotation (East 1-4, etc.)"""
        current_wind_phase = self.num_wind
        # Stay in this loop as long as the wind phase hasn't changed
        while self.num_wind == current_wind_phase and not self.is_game_over():
            self.setup_new_kyoku()
            self.play_kyoku()
            self._log_state("KYOKU OVER")
            
            # Change this to True if you implement winning logic and the dealer wins
            self.end_kyoku(dealer_renchan=False)

    def play_kyoku(self):
        """Main loop for a single hand."""
        while not self.is_kyoku_over():
            current_player = self.players[self.turn_index]

            # DRAW
            new_tile = self.wall.pop()
            self._log_state("DRAW", new_tile.to_mspzd())
            current_player.hand += new_tile

            # DISCARD
            discarded = current_player.decide_discard()
            current_player.discard_tile(discarded)
            self._log_state("DISCARD", discarded.to_mspzd())
            
            # Move to next player ONLY once at the end of the turn
            self.next_turn()

    def _log_state(self, action_type: str, tile_id: str = None):
        """Captures the current state of the table for post-game review."""
        self.players: List[MahjongPlayer]
        snapshot = {
            "turn_index": self.turn_index,
            "num_wind": self.num_wind,
            "num_kyoku": self.num_kyoku,
            "tiles_remaining": self.tiles_remaining,
            "dora": copy.deepcopy(self.dora_indicators),
            "action": action_type,
            "action_tile": tile_id,
            "players": [
                {
                    "name": p.name,
                    "score": p.score,
                    "hand": p.hand.to_ids() if p.hand else [],
                    "discards": list(p.discards),
                    "is_dealer": p.is_dealer,
                    "wind": p.wind
                } for p in self.players
            ]
        }
        self.log.append(snapshot)


"""
Ton-ba 1 Kyoku dealer=p1
p1:E, p2:S, p3:W, p4:N
Ton-ba 2 Kyoku dealer=p2
p1:N, p2:E, p3:S, p4:W
Ton-ba 3 Kyoku dealer=p3
p1:W, p2:N, p3:E, p4:S
Ton-ba 4 Kyoku dealer=p4
p1:S, p2:W, p3:N, p4:E

Nan-ba 1 Kyoku dealer=p1
p1:E, p2:S, p3:W, p4:N
Nan-ba 2 Kyoku dealer=p2
p1:N, p2:E, p3:S, p4:W
Nan-ba 3 Kyoku dealer=p3
p1:W, p2:N, p3:E, p4:S
Nan-ba 4 Kyoku dealer=p4
p1:S, p2:W, p3:N, p4:E
"""

'\nTon-ba 1 Kyoku dealer=p1\np1:E, p2:S, p3:W, p4:N\nTon-ba 2 Kyoku dealer=p2\np1:N, p2:E, p3:S, p4:W\nTon-ba 3 Kyoku dealer=p3\np1:W, p2:N, p3:E, p4:S\nTon-ba 4 Kyoku dealer=p4\np1:S, p2:W, p3:N, p4:E\n\nNan-ba 1 Kyoku dealer=p1\np1:E, p2:S, p3:W, p4:N\nNan-ba 2 Kyoku dealer=p2\np1:N, p2:E, p3:S, p4:W\nNan-ba 3 Kyoku dealer=p3\np1:W, p2:N, p3:E, p4:S\nNan-ba 4 Kyoku dealer=p4\np1:S, p2:W, p3:N, p4:E\n'

In [161]:
# VISUALIZE SIMULATION

class MahjongReplay:
    def __init__(self, history):
        self.history = history
        self.current_idx = 0
        self.output = widgets.Output()
        # Find all indices where a new round starts
        self.round_start_indices = self._get_round_indices()

    def _get_round_indices(self):
        indices = [i for i,n in enumerate(self.history) if n["action"] == "SETUP ROUND"]
        return indices

    def render(self):
        if not self.history:
            with self.output:
                print("Error: History is empty.")
            return

        state = self.history[self.current_idx]
        with self.output:
            clear_output(wait=True)
            
            # 1. Logic for Header (Wind Phase and Kyoku)
            # num_wind: 0=East Phase, 1=South Phase
            # num_kyoku: 0 to 3
            field_wind = MahjongTable.BA[state['num_wind']]
            kyoku_display_num = state['num_kyoku'] + 1
            
            # Map Dora
            dora_html = ""
            for d_val in state["dora"]:
                d_val: Hand136
                d_path_list = [n for n in mspzd_to_file_path(d_val.to_mspzd().notation) if len(n)>0]
                d_path = d_path_list[0][0] if d_path_list and d_path_list[0] else None
                if d_path:
                    dora_html += f'<img src="{d_path}" style="height: 40px; border: 1px solid #ffd700; border-radius: 2px; margin-right: 2px;">'

            html = f"""
            <div style="background-color: #1e1e1e; color: #d4d4d4; padding: 20px; border-radius: 10px; font-family: 'Consolas', monospace; border: 1px solid #333; width: fit-content; min-width: 680px;">
                <div style="border-bottom: 2px solid #4ec9b0; padding-bottom: 8px; margin-bottom: 15px; display: flex; justify-content: space-between; align-items: center;">
                    <div>
                        <span style="font-size: 1.2em; font-weight: bold; color: #4ec9b0;">{field_wind} {kyoku_display_num} Kyoku</span>
                        <span style="margin-left: 20px; color: #858585;">Tiles: {state['tiles_remaining']}</span>
                    </div>
                    <div style="display: flex; align-items: center;">
                        <span style="font-size: 0.8em; color: #ffd700; margin-right: 8px;">DORA:</span>
                        {dora_html if dora_html else '<span style="color:#444">None</span>'}
                    </div>
                </div>
            """
            
            # Handle Players
            for i, p in enumerate(state['players']):
                is_active = (i == state['turn_index'])
                active_css = "background: #252526; border: 1px solid #569cd6;" if is_active else "border: 1px solid #333;"
                
                # 1. Convert the base hand (13 tiles) to paths
                # Use p['hand'] directly since it doesn't contain the draw yet
                hand_str = MSPZD(MahjongConverter.to_str(p['hand'])).notation
                path_groups = mspzd_to_file_path(hand_str)
                all_paths = [path for group in path_groups for path in group]
                
                tile_html = ""
                # Render the base hand (typically 13 tiles)
                for path in all_paths:
                    tile_html += f'<img src="{path}" style="height: 60px; width: 38px; margin-left: 1px; border-radius: 4px; border: 1px solid #000; vertical-align: bottom;">'

                # 2. Explicitly render the Drawn tile from state['action_tile']
                if is_active and state.get('action') == "DRAW" and state.get('action_tile'):
                    draw_tile_obj = state['action_tile']
                    draw_tile_obj: MSPZD
                    draw_path_list = [n for n in mspzd_to_file_path(draw_tile_obj.notation) if len(n)>0]
                    if draw_path_list and draw_path_list[0]:
                        draw_path = draw_path_list[0][0]
                        # Add the 15px gap and a gold border to highlight the new arrival
                        tile_html += f'<img src="{draw_path}" style="height: 60px; width: 38px; margin-left: 15px; border-radius: 4px; border: 2px solid #ffd700; vertical-align: bottom;">'

                seat_wind = MahjongTable.BA[p['wind']]

                html += f"""
                <div style="padding: 12px; margin: 10px 0; border-radius: 6px; {active_css}">
                    <div style="display: flex; justify-content: space-between; margin-bottom: 8px;">
                        <span>
                            <b style="color: #9cdcfe;">[{seat_wind}]</b> {p['name']}
                            {'<span style="color: #f44747; font-size: 0.75em; font-weight: bold; margin-left: 10px; border: 1px solid #f44747; padding: 1px 4px; border-radius: 3px;">DEALER</span>' if p['is_dealer'] else ''}
                        </span>
                        <span style="color: #b5cea8; font-weight: bold;">{p['score']:,}</span>
                    </div>
                    <div style="display: flex; align-items: center; background: #111; padding: 10px; border-radius: 4px; min-height: 62px;">
                        {tile_html if tile_html else '<span style="color:#444;">No tiles</span>'}
                    </div>
                </div>
                """
            
            html += f"""
                <div style="margin-top: 10px; font-size: 0.9em; color: #ce9178; background: #2d2d2d; padding: 5px 10px; border-radius: 4px;">
                    Last Action: <b>{state['action']}</b> | Step: {self.current_idx}
                </div>
            </div>
            """
            display(HTML(html))

    # --- Navigation Logic ---
    def move_next_turn(self, _):
        if self.current_idx < len(self.history) - 1:
            self.current_idx += 1
            self.render()

    def move_prev_turn(self, _):
        if self.current_idx > 0:
            self.current_idx -= 1
            self.render()

    def move_next_round(self, _):
        for idx in self.round_start_indices:
            if idx > self.current_idx:
                self.current_idx = idx
                self.render()
                break

    def move_prev_round(self, _):
        for idx in reversed(self.round_start_indices):
            if idx < self.current_idx:
                self.current_idx = idx
                self.render()
                break

    def show(self):
        """Build and display the entire UI."""
        # Create buttons
        b1 = widgets.Button(description="PREV R", layout=widgets.Layout(width='80px'))
        b2 = widgets.Button(description="PREV T", layout=widgets.Layout(width='80px'))
        b3 = widgets.Button(description="NEXT T", layout=widgets.Layout(width='80px'))
        b4 = widgets.Button(description="NEXT R", layout=widgets.Layout(width='80px'))

        # Navigation logic (direct functions)
        def p_r(_): 
            for idx in reversed(self.round_start_indices):
                if idx < self.current_idx:
                    self.current_idx = idx; break
            self.render()
        
        def p_t(_): 
            self.current_idx = max(0, self.current_idx - 1); self.render()
            
        def n_t(_): 
            self.current_idx = min(len(self.history)-1, self.current_idx + 1); self.render()
            
        def n_r(_):
            for idx in self.round_start_indices:
                if idx > self.current_idx:
                    self.current_idx = idx; break
            self.render()

        b1.on_click(p_r); b2.on_click(p_t); b3.on_click(n_t); b4.on_click(n_r)

        # Trigger initial render
        self.render()
        
        # Display everything directly
        ui = widgets.VBox([widgets.HBox([b1, b2, b3, b4]), self.output])
        display(ui)

def mspzd_to_file_path(mspzd: str):
    result = {char: [] for char in "mpszd"}
    buffer = []

    for char in mspzd:
        if char.isdigit():
            buffer.append(char)
        elif char in result:
            result[char].extend([int(d) for d in buffer])
            result[char].sort()
            buffer = []

    # Build the 2D array in fixed order: m, p, s, z, d
    ref = "mpszd"
    paths_2d = []
    for char in ref:
        suit_paths = [
            os.path.join(TILE_IMAGE_DIR, char, f"{char}{num}.png") 
            for num in result[char]
        ]
        paths_2d.append(suit_paths)
            
    return paths_2d

In [162]:
table = MahjongTable(
    starting_score=25000,
    ending_score=30000,
    num_players = 4,
    player_names = ["sakana", "yagata", "lynn", "byron"],
    total_wind = 4
)
table.play_game()
game_log = table.log

replay = MahjongReplay(game_log)
replay.show()